# Lab 2: Structured Output & Clinical Data Extraction
## CCI Session 3

**Duration:** 15 minutes

### Clinical Scenario
> KHCC's cancer registry team receives oncology notes in free text. They need structured data — diagnosis, stage, medications, allergies — in a database-ready format. You will build an extraction pipeline using Pydantic structured outputs.

### Objective
- Use JSON mode for basic structured output
- Build Pydantic models for clinical data schemas
- Extract structured data from 10 synthetic oncology notes

## Setup — Install and Import

In [ ]:
# Run this cell first to install required packages
!pip install -q openai pydantic

from openai import OpenAI
from pydantic import BaseModel
from typing import List, Optional
import json

## API Key Setup

In [ ]:
# === API KEY ===
# TODO: Set your OpenAI API key using Colab's userdata (recommended) or paste directly
# Option 1 (recommended): from google.colab import userdata; api_key = userdata.get('OPENAI_API_KEY')
# Option 2: api_key = "sk-..."

# TODO: Create the OpenAI client
# client = OpenAI(api_key=api_key)

## Cell 1 — JSON Mode Basics
Use `response_format={"type": "json_object"}` to get structured JSON back from the model.
This is the simplest approach — the model returns valid JSON but you define the schema in the prompt.

In [ ]:
# === CELL 1: JSON MODE ===
note = "Patient Ahmad, MRN 1889, diagnosed with Stage IIIA non-small cell lung cancer. Started on carboplatin/pemetrexed. Allergic to sulfa drugs."

# TODO: Make an API call with response_format={"type": "json_object"}
# System prompt: "Extract patient information as JSON with keys: patient_name, mrn, diagnosis, stage, medications, allergies"
# Print the JSON response

response = None  # Replace with your API call

# TODO: Parse and pretty-print the JSON
# Hint: json.loads(response.choices[0].message.content)

## Cell 2 — Pydantic Models
Define Pydantic models that describe the exact schema you want.
This gives you **type safety**, **validation**, and **auto-documentation**.

In [ ]:
# === CELL 2: PYDANTIC MODELS ===
from pydantic import BaseModel
from typing import List, Optional

# TODO: Define a Medication model with: name (str), dose (Optional[str]), frequency (Optional[str])

# TODO: Define a ClinicalExtraction model with:
#   patient_name: str
#   mrn: str
#   diagnosis: str
#   stage: Optional[str]
#   medications: List[Medication]
#   allergies: List[str]
#   procedures: List[str]
#   follow_up_date: Optional[str]
#   key_findings: List[str]

## Cell 3 — Structured Output with Pydantic
Use `client.beta.chat.completions.parse()` with `response_format=ClinicalExtraction`
to get a **Pydantic object** back — not raw JSON!

In [ ]:
# === CELL 3: PYDANTIC STRUCTURED OUTPUT ===
# TODO: Use client.beta.chat.completions.parse() with response_format=ClinicalExtraction
# Pass the same note from Cell 1
# Print the parsed result — notice it's a Pydantic object, not raw JSON!

# Hint:
# result = client.beta.chat.completions.parse(
#     model="gpt-4o-mini",
#     temperature=0.0,
#     messages=[...],
#     response_format=ClinicalExtraction
# )
# parsed = result.choices[0].message.parsed
# print(type(parsed))  # Should be ClinicalExtraction
# print(parsed.model_dump_json(indent=2))

## Cell 4 — Load Synthetic Notes
Load the 10 synthetic oncology notes. Each note has:
- `note_id`: unique identifier
- `note_type`: type of clinical document
- `text`: the raw clinical note
- `expected_extraction`: ground truth for comparison

In [ ]:
# === CELL 4: LOAD 10 SYNTHETIC NOTES ===
import json

# The synthetic notes are provided as a JSON file
# In Colab, upload the file or paste the content
# TODO: Load the 10 notes from synthetic_clinical_notes.json
# If using file upload:
#   from google.colab import files
#   uploaded = files.upload()  # upload synthetic_clinical_notes.json
#   notes = json.loads(list(uploaded.values())[0].decode())

# TODO: Print the first note's text and expected extraction
# print(f"Note ID: {notes[0]['note_id']}")
# print(f"Type: {notes[0]['note_type']}")
# print(f"\nText:\n{notes[0]['text'][:200]}...")
# print(f"\nExpected extraction:")
# print(json.dumps(notes[0]['expected_extraction'], indent=2))

## Cell 5 — Batch Extraction
Loop through all 10 notes and extract structured data using the Pydantic model.
Compare your extractions against the expected values.

In [ ]:
# === CELL 5: BATCH EXTRACTION ===
# TODO: Loop through all 10 notes
# For each note, use Pydantic structured output to extract ClinicalExtraction
# Store results in a list
# Print each extraction result

extractions = []
# for i, note_data in enumerate(notes):
#     print(f"\n{'='*60}")
#     print(f"Processing {note_data['note_id']} ({note_data['note_type']})")
#     print(f"{'='*60}")
#     
#     # TODO: Make API call with Pydantic structured output
#     # TODO: Append parsed result to extractions list
#     # TODO: Print the extraction
#     pass

# TODO: Compare extractions against expected_extraction for accuracy
# Count how many fields match for each note

## Cell 6 — Few-Shot Extraction
Use the first 2 notes as examples (with their expected extractions) as few-shot examples.
Then extract from note #3 and compare quality.

In [ ]:
# === CELL 6: FEW-SHOT EXTRACTION ===
# TODO: Take the first 2 notes as examples (with their expected extractions)
# Use them as few-shot examples in the system prompt
# Then extract from note #3
# Compare: is the few-shot result better than zero-shot?

# Hint: Build a system message like:
# "You are a clinical data extractor. Here are examples of correct extractions:
#  Note: {notes[0]['text']}
#  Extraction: {json.dumps(notes[0]['expected_extraction'])}
#  Note: {notes[1]['text']}
#  Extraction: {json.dumps(notes[1]['expected_extraction'])}"
# Then pass note #3 as the user message

## Stretch Challenge
Handle this really messy note with abbreviations:

```
pt: F.H., 55M, MRN 2201. dx CRC w/ liver mets, s/p LAR 2023. 
now on FOLFIRI/bev q2w c8. tox: gr2 diarrhea, gr1 HTN. 
allx: PCN, ASA. f/u 2wk.
```

Can your extraction pipeline handle abbreviations? Try adding instructions to the system prompt.

In [ ]:
# === STRETCH CHALLENGE ===
messy_note = """pt: F.H., 55M, MRN 2201. dx CRC w/ liver mets, s/p LAR 2023. 
now on FOLFIRI/bev q2w c8. tox: gr2 diarrhea, gr1 HTN. 
allx: PCN, ASA. f/u 2wk."""

# TODO: Extract structured data from this messy note
# Hint: Add to the system prompt: "Expand all medical abbreviations before extracting."

## KHCC Connection

This lab mirrors the **cancer registry extraction pipeline** being developed in **AIDI-DB**.
In production, structured extraction from oncology notes feeds directly into:
- Cancer registry databases
- Treatment outcome tracking
- Clinical trial matching
- Quality assurance dashboards

The Pydantic schema approach ensures that extracted data is always **valid**, **typed**, and
**database-ready** — critical for clinical data integrity.